# Zaid – Week 9-10

Regression, Classification & Clustering on the California Housing dataset (with Pipelines).

## Cell 1: Install/Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report, roc_auc_score, roc_curve, silhouette_score, davies_bouldin_score)
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
print("Libraries loaded!")

## Cell 2: Load Dataset: House Pricing

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print("Shape:", df.shape)
df.head()

## Cell 3: Handling Missing Data

In [ ]:
# We introduce missing values to demonstrate cleaning strategy
df_clean = df.copy()
np.random.seed(42)
missing_indices_1 = np.random.choice(df_clean.index, size=150, replace=False)
missing_indices_2 = np.random.choice(df_clean.index, size=100, replace=False)
df_clean.loc[missing_indices_1, 'AveRooms'] = np.nan
df_clean.loc[missing_indices_2, 'AveBedrms'] = np.nan

print("Missing values before cleaning:\n", df_clean.isnull().sum())

# Strategy: Fill numeric missing values with the median
df_clean['AveRooms'] = df_clean['AveRooms'].fillna(df_clean['AveRooms'].median())
df_clean['AveBedrms'] = df_clean['AveBedrms'].fillna(df_clean['AveBedrms'].median())

print("\nMissing values after cleaning:", df_clean.isnull().sum().sum(), "total missing values left")

## Cell 4: Feature Engineering

In [ ]:
df_clean['bedrooms_ratio'] = df_clean['AveBedrms'] / df_clean['AveRooms']
df_clean['rooms_per_person'] = df_clean['AveRooms'] / df_clean['AveOccup']
df_clean['income_per_room'] = df_clean['MedInc'] / df_clean['AveRooms']

print("Columns after feature engineering:\n", df_clean.columns.tolist())
df_clean.head()

## Cell 5: Prepare Data & Scaling For Clustering

In [ ]:
# Prepare features for clustering (exclude target 'MedHouseVal')
X_clustering = df_clean.drop(columns=['MedHouseVal'])

# We will scale data specifically for clustering in the Pipeline later
# Here we just prepare the data
X_clustering_features = X_clustering.values
print("Clustering Features Shape:", X_clustering_features.shape)

## Cell 6: Multiple Regression: Train/Test Split & Pipeline

In [ ]:
X = df_clean.drop(columns=['MedHouseVal'])
y = df_clean['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

# --- Using Pipeline for Regression ---
# This ensures scaling is done ONLY on training data, preventing data leakage
regression_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

regression_pipeline.fit(X_train, y_train)
y_pred = regression_pipeline.predict(X_test)

# Looking at coefficients (need to extract from pipeline)
# Note: Pipeline coefficients are on scaled data, but order matches X_train columns
coef_df = pd.DataFrame({
    'Feature': X_train.columns, 
    'Coefficient': regression_pipeline.named_steps['regressor'].coef_
}).sort_values(by='Coefficient', ascending=False)

print("Feature coefficients (impact on price):\n", coef_df)

## Cell 7: Regression Evaluation Metrics

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("Multiple Regression Performance:")
print("R² Score:", round(r2, 3))
print("MAE:", round(mae, 3), "(in $100,000s)")
print("RMSE:", round(rmse, 3), "(in $100,000s)")

# Visualizing predicted vs actual prices
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='teal')
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--', color='red', label='Perfect prediction line')
plt.xlabel("Actual House Price ($100k)")
plt.ylabel("Predicted House Price ($100k)")
plt.title("Multiple Regression: Actual vs Predicted House Prices")
plt.legend()
plt.show()

## Cell 8: CROSS-VALIDATION (Regression with Pipeline)

In [ ]:
# Cross-validation on the Pipeline ensures scaling is re-fitted for every fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2_scores = cross_val_score(regression_pipeline, X, y, cv=kf, scoring='r2')
cv_rmse_scores = np.sqrt(-cross_val_score(regression_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error'))

print("Cross-validation R² scores (5 folds):", np.round(cv_r2_scores, 3))
print("Mean CV R²:", round(cv_r2_scores.mean(), 3), "| Std Dev:", round(cv_r2_scores.std(), 3))
print("\nCross-validation RMSE scores (5 folds):", np.round(cv_rmse_scores, 3))
print("Mean CV RMSE:", round(cv_rmse_scores.mean(), 3))

## Cell 9: CLASSIFICATION METRICS (Pipeline)

In [ ]:
# Convert continuous price to binary label
median_price = y.median()
y_binary = (y > median_price).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# --- Using Pipeline for Classification ---
clf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=5000))
])

clf_pipeline.fit(X_train_c, y_train_c)
y_pred_c = clf_pipeline.predict(X_test_c)
y_pred_proba_c = clf_pipeline.predict_proba(X_test_c)[:, 1]

# ---- Confusion Matrix ----
cm = confusion_matrix(y_test_c, y_pred_c)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Affordable', 'Expensive'],
            yticklabels=['Affordable', 'Expensive'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Expensive vs Affordable House")
plt.show()

# ---- Precision, Recall, F1 ----
precision = precision_score(y_test_c, y_pred_c)
recall = recall_score(y_test_c, y_pred_c)
f1 = f1_score(y_test_c, y_pred_c)

print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-Score:", round(f1, 3))
print("\nFull Classification Report:\n", classification_report(y_test_c, y_pred_c, target_names=['Affordable', 'Expensive']))

# ---- ROC-AUC ----
roc_auc = roc_auc_score(y_test_c, y_pred_proba_c)
fpr, tpr, thresholds = roc_curve(y_test_c, y_pred_proba_c)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess (AUC = 0.5)')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Expensive vs Affordable House")
plt.legend()
plt.show()

print("ROC-AUC Score:", round(roc_auc, 3))

## Cell 10: Clustering: K-Means, Hierarchical, Dbscan

In [ ]:
print("\n--- CLUSTERING ANALYSIS ---")

# Helper function to plot clusters using PCA
def plot_clusters(X, labels, title, method_name):
    # Reduce to 2D for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.6, edgecolor='k')
    plt.title(f"{method_name}: Clusters (Silhouette Score = {title.split('= ')[1]})")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(scatter, label='Cluster ID')
    plt.show()

# --- 1. K-Means Clustering ---
# We test K=2 to K=6 to find the optimal number of clusters
silhouette_scores_km = []
k_range = range(2, 7)

print("\n--- K-Means Evaluation ---")
for k in k_range:
    kmeans = Pipeline([
        ('scaler', StandardScaler()),
        ('kmeans', KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    kmeans.fit(X_clustering_features)
    labels_km = kmeans.predict(X_clustering_features)
    score = silhouette_score(X_clustering_features, labels_km)
    silhouette_scores_km.append(score)
    print(f"K={k}: Silhouette Score = {score:.3f}")

# Pick best K (highest silhouette score)
best_k = k_range[np.argmax(silhouette_scores_km)]
print(f"\nOptimal K for K-Means: {best_k}")

kmeans_final = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=best_k, random_state=42, n_init=10))
])
kmeans_final.fit(X_clustering_features)
km_labels = kmeans_final.predict(X_clustering_features)
km_score = silhouette_score(X_clustering_features, km_labels)

plot_clusters(X_clustering_features, km_labels, f"Silhouette Score = {km_score:.3f}", "K-Means")

# --- 2. Hierarchical Clustering (Agglomerative) ---
print("\n--- Hierarchical Clustering Evaluation ---")
h_range = range(2, 7)
silhouette_scores_h = []

for h in h_range:
    hier_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('hier', AgglomerativeClustering(n_clusters=h))
    ])
    h_labels = hier_pipeline.fit_predict(X_clustering_features)
    score = silhouette_score(X_clustering_features, h_labels)
    silhouette_scores_h.append(score)
    print(f"Clusters={h}: Silhouette Score = {score:.3f}")

best_h = h_range[np.argmax(silhouette_scores_h)]
print(f"\nOptimal Clusters for Hierarchical: {best_h}")

hier_final = Pipeline([
    ('scaler', StandardScaler()),
    ('hier', AgglomerativeClustering(n_clusters=best_h))
])
h_labels = hier_final.fit_predict(X_clustering_features)
h_score = silhouette_score(X_clustering_features, h_labels)

plot_clusters(X_clustering_features, h_labels, f"Silhouette Score = {h_score:.3f}", "Hierarchical")

# --- 3. DBSCAN Clustering ---
print("\n--- DBSCAN Evaluation ---")
# DBSCAN doesn't use n_clusters, we tune eps and min_samples
# We try a few combinations
best_db_score = -1
best_db_eps = 0
best_db_min = 0
best_db_labels = None

# Try different eps values (DBSCAN sensitive to this)
eps_vals = [0.5, 1.0, 1.5, 2.0]
for eps in eps_vals:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_clustering_features)
    
    # Check if we have more than 1 cluster (and at least one non-noise point)
    if len(set(dbscan_labels)) > 1 and -1 not in dbscan_labels: # -1 is noise
        score = silhouette_score(X_clustering_features, dbscan_labels)
        print(f"DBSCAN (eps={eps}, min_samples=5): Silhouette Score = {score:.3f}")
        if score > best_db_score:
            best_db_score = score
            best_db_eps = eps
            best_db_labels = dbscan_labels

if best_db_labels is not None:
    print(f"\nBest DBSCAN: eps={best_db_eps}, Score={best_db_score:.3f}")
    plot_clusters(X_clustering_features, best_db_labels, f"Silhouette Score = {best_db_score:.3f}", "DBSCAN")
else:
    print("DBSCAN could not form good clusters with standard parameters (too many noise points).")
    # Fallback visualization with default DBSCAN
    dbscan_default = DBSCAN(eps=1.0, min_samples=5)
    default_labels = dbscan_default.fit_predict(X_clustering_features)
    print(f"DBSCAN (eps=1.0, min_samples=5): {np.sum(default_labels == -1)} noise points found.")
    # Plot anyway to show density
    plt.figure(figsize=(7, 6))
    pca = PCA(n_components=2)
    pca.fit(X_clustering_features)
    pca_data = pca.transform(X_clustering_features)
    plt.scatter(pca_data[:, 0], pca_data[:, 1], c=default_labels, cmap='viridis', alpha=0.6, edgecolor='k')
    plt.title("DBSCAN: Clusters (Note: -1 = Noise)")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.colorbar(label='Cluster ID (or -1 for Noise)')
    plt.show()

## Cell 11: Clustering Evaluation Metrics Summary

In [ ]:
print("\n--- Clustering Metrics Summary ---")
print(f"K-Means (k={best_k}): Silhouette Score = {km_score:.3f}, Davies-Bouldin = {davies_bouldin_score(X_clustering_features, km_labels):.3f}")
print(f"Hierarchical (k={best_h}): Silhouette Score = {h_score:.3f}, Davies-Bouldin = {davies_bouldin_score(X_clustering_features, h_labels):.3f}")
if best_db_labels is not None:
    print(f"DBSCAN (eps={best_db_eps}): Silhouette Score = {best_db_score:.3f}, Davies-Bouldin = {davies_bouldin_score(X_clustering_features, best_db_labels):.3f}")

## Cell 12: Interpretation & Conclusion

In [ ]:
print("\n--- Final Interpretation ---")
print("1. Regression Pipeline: Successfully predicted house prices with R² ≈ {:.2f}.".format(r2))
print("2. Classification Pipeline: Distinguished expensive vs. affordable homes with AUC ≈ {:.2f}.".format(roc_auc))
print("3. Clustering: Used K-Means, Hierarchical, and DBSCAN to discover natural house groupings without labels.")
print("   - K-Means found {} clusters (best silhouette = {:.3f}).".format(best_k, km_score))
print("   - Hierarchical found {} clusters (best silhouette = {:.3f}).".format(best_h, h_score))
print("   - DBSCAN found density-based clusters (eps = {}).".format(best_db_eps))

# Visualize cluster centers for K-Means (optional insight)
if best_k < 6: # Only if K is small
    km_model = kmeans_final.named_steps['kmeans']
    centers = km_model.cluster_centers_
    centers_df = pd.DataFrame(centers, columns=X.columns)
    print("\nK-Means Cluster Centers (First 3 clusters):")
    print(centers_df.head(3))

print("\n🎉 Complete ML Workflow Demo (Regression, Classification, Clustering + Pipelines) finished!")